In [1]:
!pip uninstall -y transformers huggingface_hub tokenizers tensorflow h5py numpy
!pip install numpy==1.26.4
!pip install transformers==4.41.2 huggingface_hub==0.23.4 tokenizers==0.19.1 safetensors torch
!pip install langchain langchain-core langchain-community chromadb sentence-transformers

Found existing installation: transformers 4.41.2
Uninstalling transformers-4.41.2:
  Successfully uninstalled transformers-4.41.2
Found existing installation: huggingface-hub 0.23.4
Uninstalling huggingface-hub-0.23.4:
  Successfully uninstalled huggingface-hub-0.23.4
Found existing installation: tokenizers 0.19.1
Uninstalling tokenizers-0.19.1:
  Successfully uninstalled tokenizers-0.19.1
Found existing installation: numpy 1.26.4
Uninstalling numpy-1.26.4:
  Successfully uninstalled numpy-1.26.4


  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl (15.5 MB)


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chromadb 1.5.8 requires tokenizers>=0.13.2, which is not installed.
keras 3.14.0 requires h5py, which is not installed.
sentence-transformers 5.4.1 requires huggingface-hub>=0.23.0, which is not installed.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, which is not installed.
gensim 4.3.3 requires scipy<1.14.0,>=1.7.0, but you have scipy 1.17.1 which is incompatible.
streamlit 1.37.1 requires packaging<25,>=20, but you have packaging 26.1 which is incompatible.
streamlit 1.37.1 requires pandas<3,>=1.3.0, but you have pandas 3.0.2 which is incompatible.
streamlit 1.37.1 requires protobuf<6,>=3.20, but you have protobuf 6.33.6 which is incompatible.


  Using cached transformers-4.41.2-py3-none-any.whl.metadata (43 kB)
  Using cached huggingface_hub-0.23.4-py3-none-any.whl.metadata (12 kB)
  Using cached tokenizers-0.19.1-cp312-none-win_amd64.whl.metadata (6.9 kB)
Using cached transformers-4.41.2-py3-none-any.whl (9.1 MB)
Using cached huggingface_hub-0.23.4-py3-none-any.whl (402 kB)
Using cached tokenizers-0.19.1-cp312-none-win_amd64.whl (2.2 MB)


In [1]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from langchain_core.documents import Document
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFacePipeline
from transformers import pipeline as hf_pipeline

print("✅ All imports done")

C:\Users\ishas\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\ishas\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


✅ All imports done


In [3]:
rules_path = 'cognicart/data/cleaned/association_rules.csv'
seg_path = 'cognicart/data/cleaned/rfm_with_segments.csv'

try:
    rules_df = pd.read_csv(rules_path)
    print(f"✅ Rules loaded: {rules_df.shape[0]} rows")
except FileNotFoundError:
    print("⚠️ File not found — using sample data")
    rules_df = pd.DataFrame({
        'antecedents':  ['Milk', 'Bread', 'Rice'],
        'consequents':  ['Bread', 'Butter', 'Dal'],
        'support':      [0.12, 0.08, 0.15],
        'confidence':   [0.72, 0.68, 0.80],
        'lift':         [2.4, 3.1, 2.8],
        'segment':      ['Premium', 'Budget', 'Premium']
    })

try:
    seg_df = pd.read_csv(seg_path)
    if 'Monetory' in seg_df.columns:
        seg_df = seg_df.rename(columns={'Monetory': 'Total_Spend'})
    if 'Frequency' in seg_df.columns:
        seg_df['Avg_Basket'] = seg_df['Total_Spend'] / seg_df['Frequency']
    print(f"✅ Segments loaded: {seg_df.shape[0]} rows")
except FileNotFoundError:
    print("⚠️ File not found — using sample data")
    seg_df = pd.DataFrame({
        'Customer_ID': ['C001','C002','C003'],
        'Segment':     ['Premium','Budget','Regular'],
        'Total_Spend': [8500, 2100, 4300],
        'Avg_Basket':  [850, 210, 430]
    })

✅ Rules loaded: 6341 rows
✅ Segments loaded: 500 rows


In [5]:
def rule_to_document(row):
    segment = row.get('segment', 'All')
    ant = str(row['antecedents']).replace("frozenset(","").replace(")","").replace("{","").replace("}","").replace("'","").strip()
    con = str(row['consequents']).replace("frozenset(","").replace(")","").replace("{","").replace("}","").replace("'","").strip()
    text = (f"Customers who buy [{ant}] also buy [{con}]. "
            f"Support={row['support']:.2f}, Confidence={row['confidence']:.2f}, "
            f"Lift={row['lift']:.2f}. Segment: {segment}.")
    return Document(page_content=text, metadata={'segment': str(segment), 'lift': float(row['lift'])})

rule_docs = [rule_to_document(row) for _, row in rules_df.iterrows()]

seg_docs = []
for seg, grp in seg_df.groupby('Segment'):
    avg_spend = grp['Total_Spend'].mean()
    avg_basket = grp['Avg_Basket'].mean() if 'Avg_Basket' in grp.columns else avg_spend
    text = (f"Segment '{seg}': {len(grp)} customers. "
            f"Avg spend=₹{avg_spend:,.0f}. Avg basket=₹{avg_basket:,.0f}.")
    seg_docs.append(Document(page_content=text, metadata={'segment': str(seg)}))

all_docs = rule_docs + seg_docs
print(f"✅ Documents ready: {len(all_docs)}")

✅ Documents ready: 6345


In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    model_kwargs={'device': 'cpu'}
)

vectorstore = Chroma.from_documents(
    documents=all_docs,
    embedding=embeddings,
    persist_directory='cognicart/data/cognicart_chroma_db'
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print("✅ Vectorstore and retriever ready")

C:\Users\ishas\AppData\Local\Temp\ipykernel_9564\1111424713.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


✅ Vectorstore and retriever ready


In [9]:
RAG_PROMPT = PromptTemplate(
    input_variables=['context', 'question'],
    template="""You are CogniCart, a retail recommendation assistant.
Use ONLY the context below to answer. Be concise and actionable.

Context:
{context}

Question: {question}

Answer:"""
)

pipe = hf_pipeline('text2text-generation', model='google/flan-t5-base', max_new_tokens=256)
llm = HuggingFacePipeline(pipeline=pipe)

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | RAG_PROMPT
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready")

C:\Users\ishas\AppData\Local\Temp\ipykernel_9564\2569463561.py:15: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


✅ RAG chain ready


In [11]:
answer = rag_chain.invoke("What should I recommend to a customer who buys Milk?")
print(answer)

[Tea, Bread, Butter]


In [13]:
def ask_cognicart(question, verbose=True):
    docs   = retriever.invoke(question)
    answer = rag_chain.invoke(question)

    if verbose:
        print(f"\n{'='*60}")
        print(f"Q: {question}")
        print(f"{'='*60}")
        print(f"A: {answer}")
        print(f"\n📚 Retrieved from {len(docs)} document(s):")
        for s in docs:
            print(f"  • {s.page_content[:120]}...")

    return answer, docs

ask_cognicart('A customer bought Bread and Butter. What should I recommend next?')
ask_cognicart('Which products are popular among Premium segment customers?')
ask_cognicart('Summarise the buying behaviour of Budget segment customers.')


Q: A customer bought Bread and Butter. What should I recommend next?
A: [Butter]

📚 Retrieved from 3 document(s):
  • Customers who buy [Bread, Wheat Flour] also buy [Butter]. Support=0.01, Confidence=0.30, Lift=1.81. Segment: All....
  • Customers who buy [Bread, Milk, Wheat Flour] also buy [Butter]. Support=0.01, Confidence=0.43, Lift=2.58. Segment: All....
  • Customers who buy [Bread, Wheat Flour, Biscuits] also buy [Butter]. Support=0.01, Confidence=0.51, Lift=3.03. Segment: A...

Q: Which products are popular among Premium segment customers?
A: [Tea, Chips, Cold Drink]

📚 Retrieved from 3 document(s):
  • Association Rule: Customers who buy [Tea, Chips, Cold Drink] also tend to buy [Bread]. Support=0.04, Confidence=0.70, Li...
  • Association Rule: Customers who buy [Tea, Chips, Cold Drink] also tend to buy [Bread]. Support=0.04, Confidence=0.70, Li...
  • Association Rule: Customers who buy [Tea, Chips, Cold Drink] also tend to buy [Bread]. Support=0.04, Confidence=0.70, Li...


('[Tea, Biscuits, Milk, Cold Drink] also tend to buy [Bread]. Support=0.01, Confidence=0.66, Lift=1.66. Customer Segment: All Segments.',
 [Document(metadata={'segment': 'All Segments', 'consequents': 'Bread', 'antecedents': 'Tea, Biscuits, Milk, Cold Drink', 'lift': 1.6615886540120794}, page_content='Association Rule: Customers who buy [Tea, Biscuits, Milk, Cold Drink] also tend to buy [Bread]. Support=0.01, Confidence=0.66, Lift=1.66. Customer Segment: All Segments.'),
  Document(metadata={'lift': 1.6615886540120794, 'consequents': 'Bread', 'antecedents': 'Tea, Biscuits, Milk, Cold Drink', 'segment': 'All Segments'}, page_content='Association Rule: Customers who buy [Tea, Biscuits, Milk, Cold Drink] also tend to buy [Bread]. Support=0.01, Confidence=0.66, Lift=1.66. Customer Segment: All Segments.'),
  Document(metadata={'lift': 1.6615886540120794, 'antecedents': 'Tea, Biscuits, Milk, Cold Drink', 'segment': 'All Segments', 'consequents': 'Bread'}, page_content='Association Rule: Cus

In [15]:
def segment_retriever(segment_label, k=3):
    return vectorstore.as_retriever(
        search_kwargs={'k': k, 'filter': {'segment': segment_label}}
    )

def segment_aware_query(question, segment):
    seg_retriever = segment_retriever(segment)
    seg_chain = (
        {"context": seg_retriever | format_docs, "question": RunnablePassthrough()}
        | RAG_PROMPT
        | llm
        | StrOutputParser()
    )
    answer = seg_chain.invoke(question)
    print(f'\n[Segment: {segment}] Q: {question}')
    print(f'A: {answer}')
    return answer

segment_aware_query('What cross-sell opportunities exist for this customer?', 'Premium')
segment_aware_query('Suggest budget-friendly complementary products.', 'Budget')


[Segment: Premium] Q: What cross-sell opportunities exist for this customer?
A: a customer who is looking for a product

[Segment: Budget] Q: Suggest budget-friendly complementary products.
A: a t-shirt


'a t-shirt'

In [17]:
EXPLAIN_PROMPT = PromptTemplate(
    input_variables=['rule'],
    template="""You are a retail data analyst. Explain this association rule in 2-3 plain English sentences for a business manager. Mention what action the store could take.

Rule: {rule}

Explanation:"""
)

explain_chain = EXPLAIN_PROMPT | llm | StrOutputParser()

def explain_rule(antecedent, consequent, support, confidence, lift):
    rule_text = (
        f"IF customer buys [{antecedent}] THEN they also buy [{consequent}] "
        f"with Support={support:.2f}, Confidence={confidence:.2f}, Lift={lift:.2f}"
    )
    return explain_chain.invoke({'rule': rule_text})

print("=" * 60)
print("NATURAL LANGUAGE RULE EXPLANATIONS")
print("=" * 60)

top_rules = rules_df.nlargest(3, 'lift')
for _, row in top_rules.iterrows():
    explanation = explain_rule(
        row['antecedents'], row['consequents'],
        row['support'], row['confidence'], row['lift']
    )
    print(f"\nRule: {row['antecedents']} → {row['consequents']}  (Lift={row['lift']:.2f})")
    print(f"Explanation: {explanation}")

NATURAL LANGUAGE RULE EXPLANATIONS

Rule: frozenset({'Popcorn', 'Chips'}) → frozenset({'Chocolate', 'Cold Drink', 'Namkeen'})  (Lift=9.30)
Explanation: The customer can buy the frozen set with support and confidence. The customer can also buy the frozen set with cold drink. The customer can also buy the frozen set with confidence and lift.

Rule: frozenset({'Chocolate', 'Cold Drink', 'Namkeen'}) → frozenset({'Popcorn', 'Chips'})  (Lift=9.30)
Explanation: The customer can buy the frozen set with support and confidence. The customer can also buy the frozen set with popcorn. The customer can also buy the frozen set with confidence and lift. The customer can also buy the frozen set with confidence and lift. The customer can also buy the frozen set with confidence and lift. The customer can also buy the frozen set with confidence and lift. The customer can also buy the frozen set with confidence and lift. The customer can also buy the frozen set with confidence and lift. The customer can al

In [19]:
customer_queries = [
    {'Customer_ID': 'C001', 'Segment': 'Premium',  'Last_Purchase': 'Rice, Dal, Oil'},
    {'Customer_ID': 'C002', 'Segment': 'Budget',   'Last_Purchase': 'Bread, Eggs'},
    {'Customer_ID': 'C003', 'Segment': 'Regular',  'Last_Purchase': 'Tea Powder, Sugar, Milk'},
    {'Customer_ID': 'C004', 'Segment': 'Premium',  'Last_Purchase': 'Butter, Jam'},
    {'Customer_ID': 'C005', 'Segment': 'Budget',   'Last_Purchase': 'Rice'},
]

results = []
for cust in customer_queries:
    q = (f"Customer {cust['Customer_ID']} (segment: {cust['Segment']}) "
         f"last bought: {cust['Last_Purchase']}. "
         f"What 2-3 products should we recommend them next?")
    answer, _ = ask_cognicart(q, verbose=False)
    results.append({
        'Customer_ID':        cust['Customer_ID'],
        'Segment':            cust['Segment'],
        'Last_Purchase':      cust['Last_Purchase'],
        'LLM_Recommendation': answer
    })

reco_df = pd.DataFrame(results)
print(reco_df.to_string(index=False))

os.makedirs('cognicart/outputs', exist_ok=True)
reco_df.to_csv('cognicart/outputs/week8_llm_recommendations.csv', index=False)
print('\n✅ Saved to cognicart/outputs/week8_llm_recommendations.csv')

Customer_ID Segment           Last_Purchase          LLM_Recommendation
       C001 Premium          Rice, Dal, Oil          [Oil, Dal, Spices]
       C002  Budget             Bread, Eggs               [Bread, Eggs]
       C003 Regular Tea Powder, Sugar, Milk [Butter, Milk, Wheat Flour]
       C004 Premium             Butter, Jam                Butter, Jam.
       C005  Budget                    Rice          [Bread, Soap, Dal]

✅ Saved to cognicart/outputs/week8_llm_recommendations.csv


In [21]:
def rag_faithfulness_score(question, answer, source_docs):
    context_text = ' '.join([d.page_content.lower() for d in source_docs])
    answer_words = set(str(answer).lower().split())
    stopwords = {'a','an','the','is','are','and','or','to','of','in','for',
                 'with','that','this','it','by','as','at','be','was','were'}
    answer_words -= stopwords
    if not answer_words:
        return 0.0
    overlap = sum(1 for w in answer_words if w in context_text)
    return round(overlap / len(answer_words), 3)

eval_questions = [
    'What product goes well with Milk?',
    'Describe Premium segment customer behavior.',
    'Which items are frequently bought with Rice?'
]

print('RAG Faithfulness Evaluation')
print('-' * 50)
for q in eval_questions:
    ans, src = ask_cognicart(q, verbose=False)
    score = rag_faithfulness_score(q, ans, src)
    print(f'Q: {q}')
    print(f'   Faithfulness Score: {score}  (1.0 = fully grounded in context)')

RAG Faithfulness Evaluation
--------------------------------------------------
Q: What product goes well with Milk?
   Faithfulness Score: 1.0  (1.0 = fully grounded in context)
Q: Describe Premium segment customer behavior.
   Faithfulness Score: 1.0  (1.0 = fully grounded in context)
Q: Which items are frequently bought with Rice?
   Faithfulness Score: 1.0  (1.0 = fully grounded in context)
